In [5]:
import pandas as pd
from pathlib import Path
from google.colab import files

uploaded = files.upload()

# Database location
file_path = Path(
    "/content/Zimbabwe_Lithium_Research_Database_Populated.xlsx"
)

# Load research tables
projects = pd.read_excel(
    file_path,
    sheet_name="Project_Registry"
)

observations = pd.read_excel(
    file_path,
    sheet_name="Project_Annual_Observations"
)

sources = pd.read_excel(
    file_path,
    sheet_name="Source_Register"
)

print("Database loaded successfully!")
print("Projects:", len(projects))
print("Observations:", len(observations))
print("Sources:", len(sources))

Saving Zimbabwe_Lithium_Research_Database_Populated.xlsx to Zimbabwe_Lithium_Research_Database_Populated.xlsx
Database loaded successfully!
Projects: 6
Observations: 39
Sources: 32


In [6]:
# Select four core PhD case studies

core_projects = projects[
    projects["Core_Case_Study"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("yes")
].copy()

core_data = observations[
    observations["Project_ID"].isin(
        core_projects["Project_ID"]
    )
].copy()

print("Core projects:", len(core_projects))
print("Core observations:", len(core_data))

display(core_projects[["Project_ID", "Project_Name"]])

Core projects: 4
Core observations: 31


,Project_ID,Project_Name
0,Zim_L1,Arcadia-Huayou Cobalt
1,Zim_L2,Bikita Minerals
2,Zim_L3,Sabi Star Lithium-Tantalum Mine
3,Zim_L4,Sandawana Lithium Mine


INVESTMENT

In [7]:
# Step 5: Extract investment observations

investment = core_data[
    core_data["Indicator"].str.contains(
        "Acquisition|Capex",
        case=False,
        na=False
    )
].copy()

# Add project names
investment = investment.merge(
    projects[["Project_ID", "Project_Name"]],
    on="Project_ID",
    how="left"
)

# Convert investment amounts to USD millions
investment["USD_Million"] = (
    pd.to_numeric(investment["Value"], errors="coerce")
    / 1_000_000
)

# Display audit table
display(investment[
    [
        "Observation_ID",
        "Project_Name",
        "Year",
        "Indicator",
        "USD_Million",
        "Actual_or_Forecast",
        "Verification_Status",
        "Source_IDs"
    ]
])

print("Total investment records:", len(investment))

,Observation_ID,Project_Name,Year,Indicator,USD_Million,Actual_or_Forecast,Verification_Status,Source_IDs
0,OBS_002,Arcadia-Huayou Cobalt,2022,Acquisition_Price_Paid,422.0,Actual,Corroborated,SRC_01; SRC_09
1,OBS_003,Arcadia-Huayou Cobalt,2023,Capex_Spent_Reported,300.0,Actual,Corroborated,SRC_09
2,OBS_006,Arcadia-Huayou Cobalt,2025,Capex_Spent_Sulphate_Plant,400.0,Actual,Corroborated,SRC_08; SRC_32
3,OBS_008,Bikita Minerals,2022,Acquisition_Price_Paid,180.0,Actual,Corroborated,SRC_01; SRC_16
4,OBS_010,Bikita Minerals,2023,Capex_Spent_Expansion,300.0,Actual,Corroborated,SRC_01
5,OBS_014,Bikita Minerals,2026,Capex_Committed,400.0,Planned,Disputed,SRC_24
6,OBS_015,Sabi Star Lithium-Tantalum Mine,2021,Acquisition_Price_Paid,76.5,Actual,Corroborated,SRC_01; SRC_02
7,OBS_019,Sabi Star Lithium-Tantalum Mine,2026,Capex_Planned_Announced,244.0,Announced,Single-source claim,SRC_11
8,OBS_026,Sandawana Lithium Mine,2026,Capex_Committed,250.0,Planned,Single-source claim,SRC_12


Total investment records: 9


In [8]:
# Step 6: Examine Arcadia investment sources

source_check = sources[
    sources["Source_ID"].isin(["SRC_08", "SRC_32"])
]

for _, source in source_check.iterrows():
    print("\n" + "=" * 60)
    print("SOURCE:", source["Source_ID"])
    print("CITATION:", source["Full_Citation"])
    print("TYPE:", source["Source_Type"])
    print("URL:", source["URL_or_Repository"])
    print("RELIABILITY:", source["Reliability_Assessment"])
    print("LIMITATIONS:", source["Potential_Bias_or_Limitation"])


SOURCE: SRC_08
CITATION: FurtherAfrica. (2026, October 13). Zimbabwe lithium export ban locks in 2027.
TYPE: Industry & Policy News
URL: https://furtherafrica.com/2026/10/13/zimbabwe-lithium-export-ban-locks-in-2027/
RELIABILITY: Medium-High - corroborates April 2026 quota resumption and October 2026 confirmation of Jan 2027 ban
LIMITATIONS: Secondary reporting of SMM and government statements

SOURCE: SRC_32
CITATION: Shanghai Metals Market (SMM) citing MMCZ. (2026, November 9). Arcadia Lithium Sulphate Plant Completion (Oct 2025), 50,000 tpa Capacity, and Apr-Sep 2026 Exports (33,000 t).
TYPE: Market Intelligence / Official Attribution
URL: SMM report (2026-11-09) citing MMCZ — corroborated by SRC_08 (FurtherAfrica citing SMM Sept 27, 2026)
RELIABILITY: Medium-High - SMM market reporting attributed to MMCZ
LIMITATIONS: Retain MMCZ/SMM attribution; independent customs audit pending


In [9]:
# Step 7: Inspect OBS_006 in detail

record = core_data[
    core_data["Observation_ID"] == "OBS_006"
]

for column in [
    "Indicator",
    "Value",
    "Actual_or_Forecast",
    "Verification_Status",
    "Method_or_Definition",
    "Researcher_Note",
    "Page_or_Table"
]:
    print(column, ":", record.iloc[0][column])

Indicator : Capex_Spent_Sulphate_Plant
Value : 400000000.0
Actual_or_Forecast : Actual
Verification_Status : Corroborated
Method_or_Definition : Reported investment in Arcadia lithium sulphate plant completed Oct 2025
Researcher_Note : Separated Oct 2025 construction completion from Apr 2026 export start
Page_or_Table : Full article


In [10]:
# Create a separate audit table
audit = investment.copy()

# Default audit status
audit["Audit_Status"] = "Pending source review"

# Flag OBS_006
audit.loc[
    audit["Observation_ID"] == "OBS_006",
    "Audit_Status"
] = "Reported investment; actual spending unverified"

# Display results
display(
    audit[
        [
            "Observation_ID",
            "Project_Name",
            "USD_Million",
            "Verification_Status",
            "Audit_Status"
        ]
    ]
)

,Observation_ID,Project_Name,USD_Million,Verification_Status,Audit_Status
0,OBS_002,Arcadia-Huayou Cobalt,422.0,Corroborated,Pending source review
1,OBS_003,Arcadia-Huayou Cobalt,300.0,Corroborated,Pending source review
2,OBS_006,Arcadia-Huayou Cobalt,400.0,Corroborated,Reported investment; actual spending unverified
3,OBS_008,Bikita Minerals,180.0,Corroborated,Pending source review
4,OBS_010,Bikita Minerals,300.0,Corroborated,Pending source review
5,OBS_014,Bikita Minerals,400.0,Disputed,Pending source review
6,OBS_015,Sabi Star Lithium-Tantalum Mine,76.5,Corroborated,Pending source review
7,OBS_019,Sabi Star Lithium-Tantalum Mine,244.0,Single-source claim,Pending source review
8,OBS_026,Sandawana Lithium Mine,250.0,Single-source claim,Pending source review


In [11]:
# Step 9: Examine disputed Bikita investment

record = core_data[
    core_data["Observation_ID"] == "OBS_014"
]

for column in [
    "Indicator",
    "Value",
    "Actual_or_Forecast",
    "Verification_Status",
    "Method_or_Definition",
    "Researcher_Note",
    "Source_IDs"
]:
    print(column, ":", record.iloc[0][column])

# Examine the registered source
source = sources[
    sources["Source_ID"] == "SRC_24"
]

display(source[
    [
        "Full_Citation",
        "URL_or_Repository",
        "Reliability_Assessment",
        "Potential_Bias_or_Limitation"
    ]
])

Indicator : Capex_Committed
Value : 400000000.0
Actual_or_Forecast : Planned
Verification_Status : Disputed
Method_or_Definition : Under construction; Q2 2027 commissioning target (NOT yet spent)
Researcher_Note : Other company statements cite $500M total / 100,000 tpa full capacity
Source_IDs : SRC_24


,Full_Citation,URL_or_Repository,Reliability_Assessment,Potential_Bias_or_Limitation
23,"Al Jazeera. (2026, June 18). Who benefits from...",https://www.aljazeera.com/news/2026/6/18/who-b...,High - reputable international outlet with dir...,Note: $400M vs $500M Bikita sulphate plant fig...


In [12]:
# Flag the disputed Bikita investment

audit.loc[
    audit["Observation_ID"] == "OBS_014",
    "Audit_Status"
] = "Disputed: $400M vs $500M; planned, not spent"

# Display the two flagged records
display(
    audit[
        audit["Observation_ID"].isin(
            ["OBS_006", "OBS_014"]
        )
    ][
        [
            "Observation_ID",
            "Project_Name",
            "USD_Million",
            "Audit_Status"
        ]
    ]
)

,Observation_ID,Project_Name,USD_Million,Audit_Status
2,OBS_006,Arcadia-Huayou Cobalt,400.0,Reported investment; actual spending unverified
5,OBS_014,Bikita Minerals,400.0,"Disputed: $400M vs $500M; planned, not spent"


In [13]:
# Step 11: Review single-source investment claims

records = core_data[
    core_data["Observation_ID"].isin(
        ["OBS_019", "OBS_026"]
    )
]

for _, record in records.iterrows():
    print("\n" + "=" * 55)
    print("OBSERVATION:", record["Observation_ID"])
    print("INDICATOR:", record["Indicator"])
    print("VALUE:", record["Value"])
    print("STATUS:", record["Actual_or_Forecast"])
    print("VERIFICATION:", record["Verification_Status"])
    print("DEFINITION:", record["Method_or_Definition"])
    print("NOTE:", record["Researcher_Note"])
    print("SOURCES:", record["Source_IDs"])

# Display registered sources
display(
    sources[
        sources["Source_ID"].isin(
            ["SRC_11", "SRC_12"]
        )
    ][
        [
            "Source_ID",
            "Full_Citation",
            "URL_or_Repository",
            "Reliability_Assessment",
            "Potential_Bias_or_Limitation"
        ]
    ]
)


OBSERVATION: OBS_019
INDICATOR: Capex_Planned_Announced
VALUE: 244000000.0
STATUS: Announced
VERIFICATION: Single-source claim
DEFINITION: Announced Aug 24, 2026; 19-month construction (NOT yet spent)
NOTE: Planned implementation by Max Mind Investments (Zimbabwe)
SOURCES: SRC_11

OBSERVATION: OBS_026
INDICATOR: Capex_Committed
VALUE: 250000000.0
STATUS: Planned
VERIFICATION: Single-source claim
DEFINITION: Funding secured; construction mid-2026 (NOT yet spent)
NOTE: Revised down from $270M; plus $36M roads/power/water planned
SOURCES: SRC_12


,Source_ID,Full_Citation,URL_or_Repository,Reliability_Assessment,Potential_Bias_or_Limitation
10,SRC_11,"EnergyTrend. (2026, August 25). 3.2 Billion Yu...",https://www.energytrend.com/news/20260825-5201...,High - based on Shenzhen-listed Chengxin Lithi...,Planned capex/capacity (19-month construction)...
11,SRC_12,"Mapuranga, R. (2026, February 18). Sandawana P...",https://miningzimbabwe.com/sandawana-productio...,Medium - official state-owned enterprise execu...,FY2026 figures are management forecasts; earli...


In [14]:
audit.loc[
    audit["Observation_ID"] == "OBS_019",
    "Audit_Status"
] = "Announced investment; expenditure not verified"

audit.loc[
    audit["Observation_ID"] == "OBS_026",
    "Audit_Status"
] = "Funding reported; actual expenditure not verified"

display(audit[
    ["Observation_ID", "Project_Name",
     "USD_Million", "Audit_Status"]
])

print("Flagged:", (
    audit["Audit_Status"] != "Pending source review"
).sum())

print("Pending:", (
    audit["Audit_Status"] == "Pending source review"
).sum())

,Observation_ID,Project_Name,USD_Million,Audit_Status
0,OBS_002,Arcadia-Huayou Cobalt,422.0,Pending source review
1,OBS_003,Arcadia-Huayou Cobalt,300.0,Pending source review
2,OBS_006,Arcadia-Huayou Cobalt,400.0,Reported investment; actual spending unverified
3,OBS_008,Bikita Minerals,180.0,Pending source review
4,OBS_010,Bikita Minerals,300.0,Pending source review
5,OBS_014,Bikita Minerals,400.0,"Disputed: $400M vs $500M; planned, not spent"
6,OBS_015,Sabi Star Lithium-Tantalum Mine,76.5,Pending source review
7,OBS_019,Sabi Star Lithium-Tantalum Mine,244.0,Announced investment; expenditure not verified
8,OBS_026,Sandawana Lithium Mine,250.0,Funding reported; actual expenditure not verified


Flagged: 4
Pending: 5


In [15]:
# Step 13: Review remaining investment records

pending = audit[
    audit["Audit_Status"] == "Pending source review"
]

for _, record in pending.iterrows():
    print("\n" + "=" * 60)
    print("OBSERVATION:", record["Observation_ID"])
    print("PROJECT:", record["Project_Name"])
    print("AMOUNT (USD Million):", record["USD_Million"])
    print("INDICATOR:", record["Indicator"])
    print("DEFINITION:", record["Method_or_Definition"])
    print("RESEARCH NOTE:", record["Researcher_Note"])
    print("SOURCE IDs:", record["Source_IDs"])


OBSERVATION: OBS_002
PROJECT: Arcadia-Huayou Cobalt
AMOUNT (USD Million): 422.0
INDICATOR: Acquisition_Price_Paid
DEFINITION: Completed M&A acquisition price paid by Zhejiang Huayou Cobalt
RESEARCH NOTE: Separate from post-acquisition capex
SOURCE IDs: SRC_01; SRC_09

OBSERVATION: OBS_003
PROJECT: Arcadia-Huayou Cobalt
AMOUNT (USD Million): 300.0
INDICATOR: Capex_Spent_Reported
DEFINITION: Reported investment in lithium concentrator commissioned July 2023
RESEARCH NOTE: Reclassified from verified fact to reported company investment unless independently audited
SOURCE IDs: SRC_09

OBSERVATION: OBS_008
PROJECT: Bikita Minerals
AMOUNT (USD Million): 180.0
INDICATOR: Acquisition_Price_Paid
DEFINITION: Sinomine acquisition price announced Feb 2022
RESEARCH NOTE: Separate from post-acquisition expansion capex
SOURCE IDs: SRC_01; SRC_16

OBSERVATION: OBS_010
PROJECT: Bikita Minerals
AMOUNT (USD Million): 300.0
INDICATOR: Capex_Spent_Expansion
DEFINITION: Reported expenditure on 2023 expansio

In [16]:
# Step 14: Update remaining audit statuses

updates = {
    "OBS_002": "Acquisition price; source confirmation pending",
    "OBS_003": "Reported capex; independent audit pending",
    "OBS_008": "Acquisition price; completion confirmation pending",
    "OBS_010": "Reported expansion capex; independent audit pending",
    "OBS_015": "51% stake acquisition; source confirmation pending"
}

for obs_id, status in updates.items():
    audit.loc[
        audit["Observation_ID"] == obs_id,
        "Audit_Status"
    ] = status

# Check audit progress
display(
    audit[
        [
            "Observation_ID",
            "Project_Name",
            "USD_Million",
            "Audit_Status"
        ]
    ]
)

print(
    "Records awaiting initial classification:",
    (audit["Audit_Status"] == "Pending source review").sum()
)

print("Total audit records:", len(audit))


,Observation_ID,Project_Name,USD_Million,Audit_Status
0,OBS_002,Arcadia-Huayou Cobalt,422.0,Acquisition price; source confirmation pending
1,OBS_003,Arcadia-Huayou Cobalt,300.0,Reported capex; independent audit pending
2,OBS_006,Arcadia-Huayou Cobalt,400.0,Reported investment; actual spending unverified
3,OBS_008,Bikita Minerals,180.0,Acquisition price; completion confirmation pen...
4,OBS_010,Bikita Minerals,300.0,Reported expansion capex; independent audit pe...
5,OBS_014,Bikita Minerals,400.0,"Disputed: $400M vs $500M; planned, not spent"
6,OBS_015,Sabi Star Lithium-Tantalum Mine,76.5,51% stake acquisition; source confirmation pen...
7,OBS_019,Sabi Star Lithium-Tantalum Mine,244.0,Announced investment; expenditure not verified
8,OBS_026,Sandawana Lithium Mine,250.0,Funding reported; actual expenditure not verified


Records awaiting initial classification: 0
Total audit records: 9


In [17]:
# Step 15: Export investment audit

filename = "Zimbabwe_Lithium_Investment_Evidence_Audit.csv"

audit.to_csv(filename, index=False)

print("Audit file saved:", filename)
print("Total records:", len(audit))

Audit file saved: Zimbabwe_Lithium_Investment_Evidence_Audit.csv
Total records: 9


In [18]:
from google.colab import files

files.download("Zimbabwe_Lithium_Investment_Evidence_Audit.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>